# Which dashboard insights survive a statistics test?

The course capstone, `StudentPerformanceDashboard_Sanjana_Maini.xlsx`, summarises 70 students with KPI
cards, pivot charts and five written insights. A dashboard turns small differences into confident
sentences, so this notebook checks each sentence: is the difference larger than chance alone would
produce among 70 students?

Method: each claim's number is recomputed from the `Data` sheet, then compared with a permutation test
(shuffle the group labels 10,000 times and count how often a gap at least as large appears by chance),
with confidence intervals where they help. Student names are never printed.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
rng = np.random.default_rng(20261007)
R = {}
d = pd.read_excel(ROOT / "StudentPerformanceDashboard_Sanjana_Maini.xlsx", sheet_name="Data ")
d = d.drop(columns=["Student Name"])
R["students"] = len(d)
R["academic_years"] = sorted(d["Academic Year"].unique().tolist())
print(f"{len(d)} students; academic year {R['academic_years']}; departments {d['Department'].value_counts().to_dict()}; "
      f"courses {d['Course Name'].nunique()}")


def perm_spread(values, groups, n=10000):
    """p-value for the spread (max minus min) of group means under random relabelling."""
    v, g = np.asarray(values, float), np.asarray(groups)
    labels = np.unique(g)

    def spread(gg):
        m = [v[gg == lab].mean() for lab in labels]
        return max(m) - min(m)
    obs = spread(g)
    null = np.array([spread(rng.permutation(g)) for _ in range(n)])
    return obs, float((null >= obs).mean())

70 students; academic year ['2024-2025']; departments {'BCOM': 26, 'BBA': 23, 'MBA': 21}; courses 11


## Insight 1: "MBA Department has the highest average marks"

In [2]:
dep = d.groupby("Department")["Total Marks"].agg(["size", "mean", "std"]).round(2)
obs, p = perm_spread(d["Total Marks"], d["Department"])
R["insight1"] = {"means": dep["mean"].to_dict(), "spread": obs, "perm_p": p}
print(dep.to_string())
print(f"gap between highest and lowest department mean: {obs:.2f} marks; permutation p = {p:.3f}")

            size   mean   std
Department                   
BBA           23  79.91  8.32
BCOM          26  78.35  7.96
MBA           21  81.29  7.80
gap between highest and lowest department mean: 2.94 marks; permutation p = 0.430


True as arithmetic (MBA 81.3 against 78.4 for BCOM), but a gap of 2.9 marks between three groups of
21 to 26 students appears by chance 43% of the time: no evidence that MBA students perform better.

## Insight 2: "Mr. Nair gave the highest average internship score"

In [3]:
men = d.groupby("Mentor Name")["Internship Score"].agg(["size", "mean"]).round(2)
obs, p = perm_spread(d["Internship Score"], d["Mentor Name"])
R["insight2"] = {"means": men["mean"].to_dict(), "spread": obs, "perm_p": p}
print(men.to_string())
print(f"gap between highest and lowest mentor mean: {obs:.2f} points (scale 6 to 10); permutation p = {p:.3f}")

             size  mean
Mentor Name            
Mr. Iyer       17  8.18
Mr. Nair       10  8.70
Mr. Sharma     14  7.43
Mrs. Verma     10  8.30
Ms. Patel      19  7.89
gap between highest and lowest mentor mean: 1.27 points (scale 6 to 10); permutation p = 0.143


Mr. Nair's 8.7 is the highest mean, from 10 students; random assignment of the same scores to mentors
produces a gap this wide 14% of the time. Suggestive at most.

## Insight 3: "Equal number of male and female students have scored A+"

In [4]:
g = d.groupby("Gender")["Grade"].agg(n="size", a_plus=lambda s: int((s == "A+").sum()))
g["rate"] = g["a_plus"] / g["n"]
g["ci"] = [tuple(np.round(proportion_confint(k, n, method="wilson"), 3)) for k, n in zip(g["a_plus"], g["n"])]
tab = [[g.loc["F", "a_plus"], g.loc["F", "n"] - g.loc["F", "a_plus"]], [g.loc["M", "a_plus"], g.loc["M", "n"] - g.loc["M", "a_plus"]]]
fisher_p = float(stats.fisher_exact(tab)[1])
R["insight3"] = {"counts": g["a_plus"].to_dict(), "n": g["n"].to_dict(), "rates": g["rate"].round(4).to_dict(), "fisher_p": fisher_p}
print(g.to_string())
print(f"Fisher exact test on the rates: p = {fisher_p:.3f}")

         n  a_plus      rate              ci
Gender                                      
F       39       5  0.128205  (0.056, 0.267)
M       31       5  0.161290  (0.071, 0.326)
Fisher exact test on the rates: p = 0.741


The counts are equal (5 and 5) but the groups are not (39 women, 31 men), so the rates differ, 12.8%
against 16.1%; and that difference is itself well within chance (p = 0.74). The right sentence is
"A+ rates are similar for men and women", not that the counts are equal.

## Insight 4: "BCOM Department has highest number of placed students"

In [5]:
pl = d.groupby("Department")["Placement Status"].agg(n="size", placed=lambda s: int((s == "Placed").sum()))
pl["rate"] = pl["placed"] / pl["n"]
pl["ci"] = [tuple(np.round(proportion_confint(k, n, method="wilson"), 3)) for k, n in zip(pl["placed"], pl["n"])]
chi = stats.chi2_contingency(np.column_stack([pl["placed"], pl["n"] - pl["placed"]]))
R["insight4"] = {"placed": pl["placed"].to_dict(), "n": pl["n"].to_dict(), "rate": pl["rate"].round(4).to_dict(), "chi2_p": float(chi[1])}
print(pl.to_string())
print(f"chi-squared test of equal placement rates: p = {chi[1]:.3f}")

             n  placed      rate              ci
Department                                      
BBA         23       8  0.347826  (0.188, 0.551)
BCOM        26      15  0.576923  (0.389, 0.745)
MBA         21       8  0.380952  (0.208, 0.591)
chi-squared test of equal placement rates: p = 0.216


BCOM has the most placed students partly because it is the largest department. Its placement rate
(57.7%) is the highest too, but the three rates are not distinguishable at this size (p = 0.22).

## Insight 5: "Accounting has the least average total marks but the highest attendance %"

In [6]:
cr = d.groupby("Course Name").agg(n=("Total Marks", "size"), marks=("Total Marks", "mean"), attendance=("Attendance (%)", "mean")).round(2)
print(cr.sort_values("marks").to_string())
acc = d[d["Course Name"] == "Accounting"]
R["insight5"] = {"accounting_n": len(acc), "courses": int(len(cr)), "median_course_size": float(cr["n"].median())}
r_, p_ = stats.pearsonr(d["Attendance (%)"], d["Total Marks"])
R["attendance_marks_r"], R["attendance_marks_p"] = float(r_), float(p_)
print(f"Accounting has {len(acc)} students; courses have a median of {cr['n'].median():.0f} students each")
print(f"across all 70 students, attendance and total marks correlate at r = {r_:.2f} (p = {p_:.2f})")

                     n  marks  attendance
Course Name                              
Accounting           3  73.33       91.67
Economics            5  76.80       78.00
Digital Marketing    7  77.43       87.00
HR                   5  78.60       82.20
Banking             10  79.60       83.60
Taxation             8  79.62       89.00
Business Analytics   4  80.50       82.50
Entrepreneurship     2  80.50       83.00
Operations           7  81.43       86.14
Finance             13  81.46       80.31
Marketing            6  83.00       85.00
Accounting has 3 students; courses have a median of 6 students each
across all 70 students, attendance and total marks correlate at r = -0.02 (p = 0.88)


The Accounting insight rests on three students. Course-level averages from groups of 2 to 13 cannot
support a sentence; and across all 70 students, attendance has no relation to marks at all (r = -0.02),
so a course combining high attendance with low marks is exactly what chance produces.

## How many students would the dashboard need?

To detect a 3-mark difference between two groups (marks have a standard deviation of about 8), with the
usual 5% false-alarm rate and 80% power, each group needs about
$2 (z_{0.975} + z_{0.8})^2 \sigma^2 / \delta^2$ students.

In [7]:
sd = float(d["Total Marks"].std())
need = 2 * (stats.norm.ppf(0.975) + stats.norm.ppf(0.8)) ** 2 * sd ** 2 / 3 ** 2
R["sd_total_marks"], R["n_per_group_for_3_marks"] = sd, float(need)
print(f"standard deviation of total marks {sd:.1f}; students needed per group to detect 3 marks: {need:.0f}")

standard deviation of total marks 8.0; students needed per group to detect 3 marks: 112


## Conclusions

| Dashboard insight | Verdict |
|---|---|
| MBA has the highest average marks | Arithmetic, not evidence: 2.9 marks, p = 0.43 |
| Mr. Nair gives the highest internship scores | Suggestive at most: p = 0.14, 10 students |
| Equal numbers of men and women scored A+ | Counts equal, groups not; rates 12.8% vs 16.1%, no real difference |
| BCOM has the most placed students | Partly size; rates not distinguishable (p = 0.22) |
| Accounting: lowest marks, highest attendance | Three students; attendance and marks are unrelated (r = -0.02) |

With 70 students, detecting even a 3-mark difference between two groups would need about 112 students
per group. The dashboard's KPI cards are fine; its written insights should either carry sample sizes
and uncertainty or describe the data without ranking groups. (The dashboard title also says 2023-24;
every row is 2024-2025.)

In [8]:
(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "results" / "insights_tested.json").write_text(json.dumps(R, indent=1, default=float))
print(json.dumps({k: v for k, v in R.items() if not isinstance(v, dict)}, indent=1, default=float))

{
 "students": 70,
 "academic_years": [
  "2024-2025"
 ],
 "attendance_marks_r": -0.01763335441194268,
 "attendance_marks_p": 0.8848008374395866,
 "sd_total_marks": 8.006674234545617,
 "n_per_group_for_3_marks": 111.81484819070228
}
